In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
print(torch.__version__)
print(torchvision.__version__)

2.10.0+cu128
0.25.0+cu128


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

## getting dataset

In [ ]:
from pathlib import Path
from torchvision import datasets, transforms

DATA_DIR = Path(r"C:\Users\rajag_gc74h6t\OneDrive\Desktop\Deep_learning\cnn\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_data = datasets.FashionMNIST(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=transforms.ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transforms.ToTensor(),
    target_transform=None
)

print(f"Data root: {DATA_DIR}")
print(f"Torchvision dataset folder: {DATA_DIR / 'FashionMNIST'}")
print(f"Train samples: {len(train_data)} | Test samples: {len(test_data)}")


In [ ]:
import torch

SAVE_DIR = DATA_DIR / "saved_tensors"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_images = torch.stack([img for img, _ in train_data])
train_labels = torch.tensor([label for _, label in train_data])
test_images = torch.stack([img for img, _ in test_data])
test_labels = torch.tensor([label for _, label in test_data])

torch.save(train_images, SAVE_DIR / "train_images.pt")
torch.save(train_labels, SAVE_DIR / "train_labels.pt")
torch.save(test_images, SAVE_DIR / "test_images.pt")
torch.save(test_labels, SAVE_DIR / "test_labels.pt")

print(f"Saved tensors to: {SAVE_DIR}")
print(f"train_images: {train_images.shape} | train_labels: {train_labels.shape}")
print(f"test_images: {test_images.shape} | test_labels: {test_labels.shape}")
print("Saved files:")
for file_path in sorted(SAVE_DIR.glob("*.pt")):
    print(f"- {file_path.name}: {file_path.stat().st_size:,} bytes")


In [ ]:
img,label = train_data[0]
print(img.shape)
print(label)

In [ ]:
class_name = train_data.classes
cls_ind = train_data.class_to_idx
print(class_name)
print(cls_ind)
print(train_data.targets)

## visualize the sample data

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(img.squeeze(),cmap="gray")
plt.axis(False)

In [ ]:
torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))
rows,cols = 4,4
for i in range(1,rows*cols+1):
    rand_ind = torch.randint(0,len(train_data),size=[1]).item()
    print(rand_ind)
    image,labels = train_data[rand_ind]
    fig.add_subplot(rows,cols,i)
    plt.imshow(image.squeeze(),cmap="gray")
    plt.title(class_name[labels])
    plt.axis(False)


## dataloader and minibatches

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_data_loader = DataLoader(dataset = train_data,
                               batch_size = BATCH_SIZE,
                               shuffle = True)

test_data_loader = DataLoader(dataset = test_data,
                              batch_size = BATCH_SIZE,
                              shuffle =False)

print(train_data_loader)
print(test_data_loader)
train_feat,train_label = next(iter(train_data_loader))
torch.manual_seed(42)
ran_ind = torch.randint(0,len(train_feat),size=[1]).item()
image1,label1 =  train_feat[ran_ind],train_label[ran_ind]
plt.imshow(image1.squeeze(),cmap="gray")
plt.title(class_name[label1])
plt.axis(False)
print(image1.shape)

## model

In [ ]:
class fashionMNISTmodel(nn.Module):
    def __init__(self,
                 input_size:int,
                 hidden_units:int,
                 output_size:int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_size,out_features=hidden_units),
            nn.Linear(in_features=hidden_units,out_features=output_size)
        )

    def forward(self,x):
        return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)
model = fashionMNISTmodel(input_size=784,
                          hidden_units=10,
                          output_size=len(class_name))

print(model)

In [ ]:
dummy_x = torch.rand([1,1,28,28])
print(model(dummy_x))
print(model.state_dict())

## loss function and optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(),lr=0.1)

## tracking model performance and speed

In [ ]:
from timeit import default_timer as timer

def print_train_time(start:float,
                     end:float,
                     device:torch.device=None):
    total_time= end - start
    print(f"total time:{total_time:.3f},device name :{device}")
    return total_time

In [ ]:
start =timer()
print("hi")
end =timer()
print(print_train_time(start=start,end=end,device="cpu"))

## create training loop and train model on batches

In [ ]:
from tqdm.auto import tqdm

def accuracy_func(y_true, y_pred):
    correct = torch.eq(y_true, y_pred.argmax(dim=1)).sum().item()
    acc = (correct / len(y_true)) * 100
    return acc

torch.manual_seed(42)
train_start = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"epoch:{epoch}\n----")
    train_loss = 0
    train_acc = 0

    for batch, (x, y) in enumerate(train_data_loader):
        model.train()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss
        train_acc += accuracy_func(y, y_pred)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"looked at {batch * len(x)}/{len(train_data_loader.dataset)}")

    train_loss /= len(train_data_loader)
    train_acc /= len(train_data_loader)

    test_loss, test_acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for x_test, y_test in test_data_loader:
            test_pred = model(x_test)
            test_loss += loss_fn(test_pred, y_test)
            test_acc += accuracy_func(y_test, test_pred)

        test_loss /= len(test_data_loader)
        test_acc /= len(test_data_loader)

    print(f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.2f}%")
    print(f"test_loss: {test_loss:.4f} | test_acc: {test_acc:.2f}%")
end_train= timer()
train_model_time = print_train_time(start=train_start,end=end_train,device=str(next(model.parameters()).device))

In [ ]:
torch.manual_seed(42)
def eval_model(model:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               accuracy_func):
    loss,acc =0,0
    model.eval()
    with torch.inference_mode():
        for x,y in data_loader:
            y_pred1 = model(x)
            loss+=loss_fn(y_pred1,y)
            acc+= accuracy_func(y, y_pred)

        loss/= len(data_loader)
        acc/=len(data_loader)

    return {"model_name":model.__class__.__name__,
            "model_loss":loss.item(),
            "model_acc":acc}      

model_results = eval_model(model=model,
                           data_loader=test_data_loader,
                           loss_fn=loss_fn,
                           accuracy_func=accuracy_func)

print(model_results)